In [1]:
import re
import pandas as pd
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window
import sklearn
import scipy
import unicodedata 

# ---- Layer ----
CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"        # analytics layer — you have read+write here
MART    = "gms_us_mart"        # read-only for you (source data lives here)

# ---- RAW reference inputs (your uploaded tables) ----
RAW_CLINICAL = f"{CATALOG}.{ALYT}.Clinicalid_deviations"
RAW_DOCS = f"{CATALOG}.{ALYT}.documents_number_deviations"
RAW_ACRONYM  = f"{CATALOG}.{ALYT}.Acronyms_other_deviations"
RAW_CRO      = f"{CATALOG}.{ALYT}.Cro_deviations"
RAW_DEVICE   = f"{CATALOG}.{ALYT}.rd_device_list_deviations"

# ---- SOURCE deviation data (read-only, in mart) ----
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"

# ---- OUTPUT lookups (MUST be in analytics layer — you can't write to mart) ----
REF_CLINICAL = f"{CATALOG}.{ALYT}.ref_clinical_norm"
REF_DOCS     = f"{CATALOG}.{ALYT}.ref_docs_norm"
REF_ACRONYM  = f"{CATALOG}.{ALYT}.ref_acronym_norm"
REF_CRO      = f"{CATALOG}.{ALYT}.ref_cro_norm"
REF_DEVICE   = f"{CATALOG}.{ALYT}.ref_device_norm"
REF_UNIFIED  = f"{CATALOG}.{ALYT}.ref_glossary_unified"

# ---- The contract every lookup MUST expose (extra cols allowed) ----
LOOKUP_SCHEMA = ["key_norm", "entity_type", "canonical_id", "enrichment_text"]

# Fuzzy-match acceptance threshold for CRO/system names (0-100 rapidfuzz scale)
CRO_FUZZY_THRESHOLD = 90
DEVICE_FUZZY_THRESHOLD = 90

print("Config loaded.")
print("  Inputs :", RAW_CLINICAL, RAW_DOCS, RAW_ACRONYM, RAW_CRO, RAW_DEVICE, sep="\n           ")
print("  Source :", SOURCE_TABLE)
print("  Output :", REF_UNIFIED)

Config loaded.
  Inputs :
           us_gmsgq_dev.gms_us_alyt.Clinicalid_deviations
           us_gmsgq_dev.gms_us_alyt.documents_number_deviations
           us_gmsgq_dev.gms_us_alyt.Acronyms_other_deviations
           us_gmsgq_dev.gms_us_alyt.Cro_deviations
           us_gmsgq_dev.gms_us_alyt.rd_device_list_deviations
  Source : us_gmsgq_dev.gms_us_mart.tw_deviation_data_formatted_rdq
  Output : us_gmsgq_dev.gms_us_alyt.ref_glossary_unified


In [2]:
# ============================================================================
# SECTION A — CLINICAL IDs  (extract + parse; 3-source logic: free text + protocol + program)
# ============================================================================
CLINICAL_ID_ALTERNATIVES = [
    r"TAK[-\s_]?\d{2,4}[-_/]\d{3,4}",
    r"TAK[-\s_]?\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]CCT-\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"MLN[-\s_]?\d{3,4}",
    r"SHP[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"SHP[-\s_]?\d{3,4}",
    r"HGT[-\s_]?[A-Z]{2,4}[-_/]\d{2,4}",
    r"CCT[-_]?\d{2,4}",
    r"DEN[-_]?\d{2,4}",
    r"C\d{5}",
]
ID_REGEX = re.compile("|".join(CLINICAL_ID_ALTERNATIVES), flags=re.IGNORECASE)

# Program_Number filter: keep only values whose leading token is a valid clinical-ID shape.
PROGRAM_FILTER_STR = (
    r"^(TAK-?\d{2,4}|MLN\d{4}|SHP-?\d{3,4}|HGT-[A-Z]{2,4}-\d{2,4}"
    r"|C\d{5}|CCT-\d{2,4}|DEN-\d{2,4})"
)

def _which_prefix(u):
    for p in ("TAK", "MLN", "SHP", "HGT", "CCT", "DEN"):
        if u.startswith(p):
            return p
    if re.match(r"C\d{5}$", u):
        return "INTERNAL"
    return "UNKNOWN"

def parse_clinical_id(raw):
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_/]+", "-", str(raw).upper())).strip("-")
    prefix = _which_prefix(u)
    compound = suffix = None
    if prefix in ("TAK", "MLN", "SHP"):
        m = re.match(prefix + r"-?(\d+)(?:-(.+))?$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = (prefix + "-" + compound) if compound else u
    elif prefix == "HGT":
        m = re.match(r"HGT-([A-Z]{2,4})-(\d+)$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = ("HGT-" + compound) if compound else u
    elif prefix in ("CCT", "DEN"):
        m = re.match(prefix + r"-?(\d+)$", u)
        compound, suffix, key = prefix, (m.group(1) if m else None), u
    elif prefix == "INTERNAL":
        m = re.match(r"C(\d+)$", u)
        compound, key = (m.group(1) if m else None), u
    else:
        key = u
    return {"raw": raw, "canonical": u, "prefix": prefix,
            "compound_number": compound, "study_suffix": suffix, "normalized_key": key}

@F.udf(T.ArrayType(T.StringType()))
def extract_clinical_keys_udf(text):
    """(1) Free text: extract clinical IDs, return normalized compound+study keys."""
    if not text: return []
    keys = set()
    for m in ID_REGEX.finditer(str(text)):
        p = parse_clinical_id(m.group(0))
        if p["normalized_key"]: keys.add(p["normalized_key"])
        if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

@F.udf(T.ArrayType(T.StringType()))
def protocol_keys_udf(v):
    """(2) Study_Protocol: split on ';', TRUST ALL values -> normalized clinical keys."""
    if not v: return []
    keys = set()
    for s in re.split(r"\s*;\s*", str(v)):
        s = s.strip()
        if not s: continue
        p = parse_clinical_id(s)
        if p["normalized_key"]: keys.add(p["normalized_key"])
        if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

@F.udf(T.ArrayType(T.StringType()))
def program_keys_udf(v):
    """(3) Program_Number: split on ';', keep only valid clinical-ID patterns."""
    if not v: return []
    keys = set()
    for s in re.split(r"\s*;\s*", str(v)):
        s = s.strip()
        if s and re.match(PROGRAM_FILTER_STR, s):
            p = parse_clinical_id(s)
            if p["normalized_key"]: keys.add(p["normalized_key"])
            if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

def _strip_accents(s):
    """Fold accented Latin chars to ASCII (Genève -> Geneve). Non-Latin left as-is."""
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

def clean_freetext(v):
    """Case-PRESERVING cleanup: unicode hyphens/quotes/whitespace + accent folding.
       Do NOT lowercase — acronym & CRO extraction rely on original casing."""
    if not v:
        return None
    s = unicodedata.normalize("NFKC", str(v))
    s = re.sub(r"[\u2010-\u2015\u2212]", "-", s)   # unicode hyphens/minus -> '-'
    s = s.replace("\u00a0", " ")                    # non-breaking space -> space
    s = _strip_accents(s)                           # fold accents for matching
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip() or None

clean_freetext_udf = F.udf(clean_freetext, T.StringType())

# --- Language detection (English-only embedding model requires English text) ---
def _detect_lang(text):
    """Return ISO code ('en', 'de', ...) or None. Short/ID-only ASCII text -> 'en'.
       Text containing CJK / substantial non-Latin script is NOT auto-whitelisted."""
    if not text or not text.strip():
        return None
    stripped = text.strip()

    # If there's any CJK / non-Latin letters, fall through to real detection.
    has_cjk = re.search(
        r"[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7af]", stripped
    )

    if not has_cjk:
        # ASCII-ish: allow the code-only escape hatch
        residual = re.sub(r"\b[A-Z]{2,}[-\s]?\d+\b", " ", stripped)
        if not re.search(r"[A-Za-z]{3,}", residual):
            return "en"

    try:
        from langdetect import detect, DetectorFactory
        DetectorFactory.seed = 0
        return detect(stripped)
    except Exception:
        return None

def keep_if_english(text):
    """Return the text if English (or code-only), else None. Case-preserving."""
    lang = _detect_lang(text)
    return text if lang == "en" else None

keep_if_english_udf = F.udf(keep_if_english, T.StringType())

# ============================================================================
# SECTION B — DOCUMENTS  (regex EXACTLY matching SQL patterns)
# ============================================================================
DOC_PATTERNS = [
    r"\bSOP-\d+",
    r"\bSPEC-\d+",
    r"\bMTHD-\d+",
    r"\bMTD-\d+",      # legacy method prefix seen in your reference (MTD-002598)
    r"\bPROC-\d+",
    r"\bTOOL-\d+",
    r"\bFORM-\d+",
    r"\bWI-\d+",
]
DOC_REGEX = re.compile("|".join(DOC_PATTERNS), flags=re.IGNORECASE)
DOC_PREFIXES = r"(SOP|SPEC|MTHD|MTD|PROC|TOOL|FORM|WI)"

def norm_doc(v):
    """Canonical doc key. Note: MTD->MTHD collapse so legacy+current align."""
    if not v: return None
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_]+", "-", str(v).upper())).strip("-")
    if not re.match(DOC_PREFIXES + r"-?\d", u): return None
    u = re.sub(r"^MTD-", "MTHD-", u)          # unify legacy method prefix
    return u

norm_doc_udf = F.udf(norm_doc, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def extract_doc_keys_udf(text):
    """Extraction-side twin of your SQL: pull all doc IDs from text, normalized."""
    if not text: return []
    out = set()
    for m in DOC_REGEX.finditer(str(text)):
        k = norm_doc(m.group(0))
        if k: out.add(k)
    return list(out)

@F.udf(T.ArrayType(T.StringType()))
def legacy_doc_keys_udf(v):
    """Reference-side: parse messy legacy_numbers__c like
       'LSHIRE_1174436_6_0;;n/a;;TO SOP-0895' -> ['SOP-0895']."""
    if not v: return []
    out = set()
    for chunk in str(v).split(";;"):
        chunk = chunk.strip()
        if not chunk or chunk.lower() == "n/a": continue
        for m in re.finditer(DOC_PREFIXES + r"[-\s]?\d{3,7}", chunk, re.I):
            k = norm_doc(m.group(0))
            if k: out.add(k)
    return list(out)

# ============================================================================
# SECTION C — ACRONYMS  (regex candidate extraction + KNOWN-SET filter)
# ============================================================================
ACRONYM_CANDIDATE = re.compile(r"\(?[A-Z]{2,}\)?(?:[-/][A-Z0-9]+)?|\([A-Z]{2,}\)\s?[A-Z]{2,}")

def norm_acr(v):
    if not v: return None
    u = re.sub(r"\s+", " ", str(v).strip()).upper()
    return u or None

norm_acr_udf = F.udf(norm_acr, T.StringType())

def make_extract_acronym_udf(known_keys):
    """Factory: returns a UDF that extracts only acronyms present in the known set."""
    known = frozenset(known_keys)
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text: return []
        out = set()
        for m in ACRONYM_CANDIDATE.finditer(str(text)):
            k = norm_acr(m.group(0))
            if k and k in known:
                out.add(k)
        return list(out)
    return _udf

# ============================================================================
# SECTION D — CRO / SYSTEMS  (normalize only; matching is FUZZY at join time)
# ============================================================================
def norm_name(v):
    if not v: return None
    u = re.sub(r"[^A-Z0-9 ]", " ", str(v).upper())
    u = re.sub(r"\s+", " ", u).strip()
    return u or None

norm_name_udf = F.udf(norm_name, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def cro_variants_udf(name, disp, alias):
    """Reference-side: normalized name/displayName/alias variants for fuzzy pool."""
    return list({norm_name(x) for x in (name, disp, alias) if x and norm_name(x)})

def make_extract_cro_exact_udf(known_keys):
    """Exact-match twin of the fuzzy pass: extract candidate name spans that appear
       VERBATIM (after norm_name) in the CRO key set. Cheap first pass before fuzzy."""
    known = frozenset(known_keys)
    _cand = re.compile(r"\b([A-Z][A-Za-z0-9]+(?:\s+[A-Z][A-Za-z0-9]+){0,3}|[A-Z]{2,})\b")
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text: return []
        out = set()
        for m in _cand.finditer(str(text)):
            k = norm_name(m.group(1))
            if k and k in known:
                out.add(k)
        return list(out)
    return _udf


# ============================================================================
# SECTION E — DEVICES  (substring match on nonsensical names; context = type + owner)
# ============================================================================
# Business-owner abbreviation expansions (longest keys first at match time)
BUSINESS_OWNER_MAP = {
    "RGH TAU": "Rare Genetics and Hematology Therapeutic Area Unit",
    "GI TAU":  "Gastrointestinal Therapeutic Area Unit",
    "OTAU":    "Oncology Therapeutic Area Unit",
    "PDT":     "Plasma Derived Therapy",
}

def expand_business_owner(v):
    """Expand known business-owner abbreviations found anywhere in the value."""
    if v is None or str(v).strip() == "":
        return ""
    s = str(v)
    # replace longer keys first to avoid partial shadowing
    for abbr in sorted(BUSINESS_OWNER_MAP, key=len, reverse=True):
        s = re.sub(rf"\b{re.escape(abbr)}\b", BUSINESS_OWNER_MAP[abbr], s, flags=re.I)
    return s
expand_business_owner_udf = F.udf(expand_business_owner, T.StringType())

def norm_device(v):
    """Lowercase + collapse whitespace. Device names are matched as substrings,
       so we keep them permissive (no aggressive stripping)."""
    if not v:
        return None
    u = re.sub(r"\s+", " ", str(v)).strip().lower()
    return u or None

norm_device_udf = F.udf(norm_device, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def device_variants_udf(name, alias, long_name):
    """Reference-side: normalized name/alias/long_name variants for the device pool."""
    return list({norm_device(x) for x in (name, alias, long_name) if x and norm_device(x)})

def make_extract_device_exact_udf(known_keys):
    """Substring matcher: return every device key that appears in the (lowercased) text.
       Mirrors the SQL `instr(lower(text), lower(name)) > 0` logic."""
    known = list(frozenset(k for k in known_keys if k and len(k) >= 3))  # skip 1-2 char noise
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text:
            return []
        low = str(text).lower()
        return [k for k in known if k in low]
    return _udf

print("Shared UDFs registered: clinical (3-source), doc, acronym(factory), cro (exact+fuzzy).")

Shared UDFs registered: clinical (3-source), doc, acronym(factory), cro (exact+fuzzy).


In [3]:
ac = spark.table(RAW_ACRONYM)

# Guard: don't let short acronyms shadow real clinical/doc IDs
ID_LIKE = r"^(TAK|MLN|SHP|HGT|CCT|DEN|SOP|SPEC|MTHD|FORM|TOOL|WI|PROC)[-\s]?\d"

ref_acronym = (
    ac.withColumn("key_norm", norm_acr_udf(F.col("Acronym")))
      .filter(F.col("key_norm").isNotNull())
      .filter(F.length("key_norm") >= 2)                      # drop 1-char noise
      .filter(~F.col("key_norm").rlike(ID_LIKE))              # don't shadow IDs
      .filter(F.col("Category").isin("Medical", "Industry"))  # keep relevant senses
      .groupBy("key_norm")
      .agg(
          F.concat_ws(" | ", F.collect_set("Definition")).alias("senses"),
          F.first("Acronym", ignorenulls=True).alias("canonical_id"),
      )
      .withColumn("entity_type", F.lit("ACRONYM"))
      .withColumn("enrichment_text", F.concat(F.lit("Acronym expansions: "), F.col("senses")))
      .select(*LOOKUP_SCHEMA)
)

(ref_acronym.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_ACRONYM))
spark.sql(f"COMMENT ON TABLE {REF_ACRONYM} IS "
          "'Sense-aware acronym lookup: key_norm→all definitions. Built from raw_acronym.'")

display(spark.table(REF_ACRONYM).limit(20))

# Build a plain dict {ACRONYM_UPPER: first_definition} for TA/modality expansion in Cell 4
_acr_pd = (ac.filter(F.col("Category").isin("Medical", "Industry"))
             .select("Acronym", "Definition").toPandas())
ACR_MAP = {norm_acr(a): d for a, d in zip(_acr_pd["Acronym"], _acr_pd["Definition"]) if norm_acr(a)}

# Hand-curated TA overrides win over the generic acronym table
TA_OVERRIDES = {
    "NS": "Neuroscience", "ONC": "Oncology", "GI": "Gastroenterology",
    "RARE": "Rare Diseases", "PDT": "Plasma-Derived Therapies",
}
def expand_abbr(v):
    if v is None or str(v).strip() == "":
        return ""
    u = norm_acr(v)
    return TA_OVERRIDES.get(u) or ACR_MAP.get(u) or v   # fall back to original text
expand_abbr_udf = F.udf(expand_abbr, T.StringType())

print(f"Acronym map size: {len(ACR_MAP):,}")

,key_norm,entity_type,canonical_id,enrichment_text
0,(EU) CTR,ACRONYM,(EU) CTR,Acronym expansions: (European Union) Clinical Trials Regulation
1,3PL,ACRONYM,3PL,Acronym expansions: 3rd (Third) Party Logistics Providers
2,4PL,ACRONYM,4PL,Acronym expansions: 4th (Fourth) Party Logistics Providers
3,5-ASA,ACRONYM,5-ASA,Acronym expansions: 5-aminosalicylic acid
4,6MWT,ACRONYM,6MWT,Acronym expansions: Six-minute walking test
5,AAALAC,ACRONYM,AAALAC,Acronym expansions: Association for Assessment and Accreditation of Laboratory Animal Care
6,AAAS,ACRONYM,AAAS,Acronym expansions: American Association for the Advancement of Science
7,AACR,ACRONYM,AACR,Acronym expansions: American Association for Cancer Research
8,AADA,ACRONYM,AADA,Acronym expansions: Abbreviated Antibiotic Drug Application
9,AAR,ACRONYM,AAR,Acronym expansions: After Action Review


Acronym map size: 1,667


In [4]:
# (add to end of Cell 3, after ref_acronym is built)
acr_key_set = set(r["key_norm"] for r in spark.table(REF_ACRONYM).select("key_norm").collect())
extract_acronym_keys_udf = make_extract_acronym_udf(acr_key_set)   # pass the plain set
print(f"Acronym known-set size: {len(acr_key_set):,}")

Acronym known-set size: 1,667


In [5]:
# ============================================================================
# CELL 5 — REF_CLINICAL  (match on ALL alias cols; rich expanded context)
# ============================================================================
clin = spark.table(RAW_CLINICAL)
print("RAW_CLINICAL columns:", clin.columns)   # <- verify alias/context names

# --- Alias columns: any of these, if present in text, should resolve the row ---
CLIN_ALIAS_COLS = [
    "Parent_Node", "Name", "Protocol Number", "Alternate_Name",
    "Development_Name", "Parent_Alias", "Grand_Parent", "Grandparent_Alias",
]

def _clin_c(name):
    return F.col(f"`{name}`") if name in clin.columns else F.lit(None).cast("string")

# UDF: build normalized keys from every available alias value
@F.udf(T.ArrayType(T.StringType()))
def clinical_alias_keys_udf(*vals):
    keys = set()
    for v in vals:
        if v and str(v).strip():
            p = parse_clinical_id(str(v))
            if p["normalized_key"]: keys.add(p["normalized_key"])
            if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

# --- Context (expand TA/modality abbreviations) ---
clin_enriched = clin.withColumn(
    "enrichment_text",
    F.concat_ws(
        "\n",
        F.concat(F.lit("Generic: "),          F.coalesce(_clin_c("Generic_Name"), F.lit(""))),
        F.concat(F.lit("Modality: "),         expand_abbr_udf(_clin_c("Modality"))),
        F.concat(F.lit("Therapeutic Area: "), expand_abbr_udf(_clin_c("PF_TherapeuticArea"))),
        F.concat(F.lit("Finance TA Grouping: "), expand_abbr_udf(_clin_c("finance_TA_Grouping"))),
        F.concat(F.lit("Indication: "),       F.coalesce(_clin_c("IND_DESC"), F.lit(""))),
        F.concat(F.lit("Target: "),           F.coalesce(_clin_c("Target_Long_Name"), F.lit(""))),
        F.concat(F.lit("Mechanism: "),        F.coalesce(_clin_c("Mechanism"), F.lit(""))),
    ),
).withColumn(
    "canonical_id",
    F.coalesce(_clin_c("Development_Name"), _clin_c("Name"), _clin_c("Protocol Number")),
)

alias_cols_present = [c for c in CLIN_ALIAS_COLS if c in clin.columns]
ref_clinical = (
    clin_enriched
    .withColumn("key_norm", F.explode(
        clinical_alias_keys_udf(*[F.col(f"`{c}`") for c in alias_cols_present])))
    .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
    .withColumn("entity_type", F.lit("CLINICAL_ID"))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])
)

(ref_clinical.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_CLINICAL))
spark.sql(f"COMMENT ON TABLE {REF_CLINICAL} IS "
          "'Clinical-ID lookup keyed on all alias columns; context expanded via acronym map.'")
display(spark.table(REF_CLINICAL).limit(20))

RAW_CLINICAL columns: ['Parent Node', 'Name', 'Alias', 'Alliance Partner', 'Brand Name', 'Direct Flag', 'Generic Name', 'IP Owner', 'Modality', 'Orphan Designation', 'Pass Through', 'PFP Status', 'Phase (Nominal)', 'Portfolio Status', 'Protocol Number', 'Budget_Region', 'Pool - Target Property', 'ASC_CMC_MODEL_T', 'ASC_DEV_MODEL_T', 'ASC_PRD_MODEL_T', 'PrePostCS', 'PF_TherapeuticArea', 'ProjectType', 'ASC_GAO_MODEL_I', 'PF_Research_Code', 'PRD_DDU', 'IND_CODE', 'IND_DESC', 'Time_Tracking', 'Source', 'Alternate_Name', 'Development_Name', 'Fin_Alloc_Target', 'Finance_TA_Grouping', 'Modality_Detail', 'PR_Lead_Indication', 'Origin', 'Target Long Name', 'Target_Short_Name', 'Mechanism', 'Business_Owner', 'Lead_Backup_Indicator', 'Lead_Backup_Asset_Compound', 'Indication', 'Route_of_Administration', 'NME_LCM', 'Added On', 'Protocol_Phase', 'Protocol_Status', 'Protocol_Title', 'Trial_ID', 'Trial_Type', 'Partnership_Alias_Link', 'Partnership_Name_Link', 'Partner_Name_Link', 'Parent_Alias', 'Gr

,key_norm,entity_type,canonical_id,enrichment_text
0,(RE)-ACC1-INHIBITOR-(3238-100),CLINICAL_ID,PFP-B0123300,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism:
1,(RE)ACC2-INHIBITOR-(3448-100),CLINICAL_ID,PFP-B0125900,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism:
2,(RE)GOAT-INHIBITOR-(3714-100),CLINICAL_ID,PFP-B0191600,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Neuroscience\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: Inhibitor
3,(RE)GPR40-AGONIST-(3751-100),CLINICAL_ID,PFP-B1025800,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Others\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: Agonist
4,(RE)MEK-INHIBITOR-(3738-100),CLINICAL_ID,PFP-B1023700,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Gastroenterology\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: Inhibitor
5,(RE)MGLUR5-NAM-(3636-100),CLINICAL_ID,PFP-B0143800,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Neuroscience\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: NAM
6,(RE)OREXIN2-RECEPTOR-ANTAGONIST-(3608-100),CLINICAL_ID,PFP-B0142600,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Neuroscience\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: Antagonist
7,(RE)SELECTIVE-ANDROGEN-RECEPTOR-MODULATOR(XVGEN)-(3350-110),CLINICAL_ID,PFP-B0106900,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Others\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism:
8,(RE)SPT-INHIBITOR-(3483-100),CLINICAL_ID,PFP-B0126400,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism:
9,(RE)VEGFR2-INHIBITOR-(3468-100),CLINICAL_ID,PFP-B0108600,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Others\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism:


In [6]:
# ============================================================================
# CELL 6 — REF_DOCS  (keys from doc# + previous# + legacy; context = title)
# ============================================================================
docs = spark.table(RAW_DOCS)
print("RAW_DOCS columns:", docs.columns)   # <- verify names below

DOC_NUM_COL    = "document_number__v"
DOC_PREV_COL   = "previous_document_number__c"
DOC_LEGACY_COL = "legacy_numbers__c"
DOC_TITLE_COL  = "title__v"

def _dc(name):
    return F.col(f"`{name}`") if name in docs.columns else F.lit(None).cast("string")

docs_enriched = docs.withColumn(
    "enrichment_text",
    F.concat(F.lit("Document Title: "), F.coalesce(_dc(DOC_TITLE_COL), F.lit(""))),
).withColumn("canonical_id", _dc(DOC_NUM_COL))

# straightforward single-value doc columns -> norm_doc
def _doc_keys_from(col_name):
    if col_name not in docs.columns:
        return None
    return (docs_enriched
            .withColumn("key_norm", norm_doc_udf(F.col(f"`{col_name}`")))
            .filter(F.col("key_norm").isNotNull()))

parts = [df for df in (_doc_keys_from(DOC_NUM_COL), _doc_keys_from(DOC_PREV_COL)) if df is not None]

# messy legacy column -> legacy_doc_keys_udf (explode)
if DOC_LEGACY_COL in docs.columns:
    parts.append(
        docs_enriched
        .withColumn("key_norm", F.explode(legacy_doc_keys_udf(F.col(f"`{DOC_LEGACY_COL}`"))))
        .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
    )

ref_docs_union = parts[0]
for df in parts[1:]:
    ref_docs_union = ref_docs_union.unionByName(df)

ref_docs = (
    ref_docs_union
    .withColumn("entity_type", F.lit("DOCUMENT"))
    .select(*LOOKUP_SCHEMA)
    .withColumn("_len", F.length("enrichment_text"))
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("key_norm").orderBy(F.col("_len").desc())))
    .filter(F.col("_rn") == 1)
    .select(*LOOKUP_SCHEMA)
)

(ref_docs.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_DOCS))
spark.sql(f"COMMENT ON TABLE {REF_DOCS} IS "
          "'Doc lookup keyed on document_number__v + previous + legacy; context = title__v.'")
display(spark.table(REF_DOCS).limit(20))

RAW_DOCS columns: ['document_number__v', 'previous_document_number__c', 'legacy_numbers__c', 'title__v']


,key_norm,entity_type,canonical_id,enrichment_text
0,FORM-0000009,DOCUMENT,FORM-261412,Document Title: ADS-Protocol for MLN0415 Drug Substance Batch 0415-046-RO Stability Studies
1,FORM-0000012,DOCUMENT,FORM-261457,Document Title: HPLC Method for MLN8237-004 Drug Substance; Identification; Assay and Impurity Determination
2,FORM-0000018,DOCUMENT,FORM-261459,Document Title: pH Determination
3,FORM-0000023,DOCUMENT,FORM-261463,Document Title: ADS for MLN0518 Reference Standard Batch RS-011-02 Stability Studies
4,FORM-0000024,DOCUMENT,FORM-261464,Document Title: ADS-Protocol for the Long-term and Accelerated Stability MLN518 Drup Substance Batch 0518-036-F0
5,FORM-0000029,DOCUMENT,FORM-266995,Document Title: Investigational Medicinal Product (IMP) Site to Site Transfer Request Form
6,FORM-0000030,DOCUMENT,FORM-242306,Document Title: Test Method Template
7,FORM-0000052,DOCUMENT,FORM-242308,Document Title: PDMS: Cognos Report Request and Verification Form
8,FORM-0000057,DOCUMENT,FORM-242309,Document Title: Regulated PGASys Administrative Change Request Form - A
9,FORM-0000059,DOCUMENT,FORM-242310,Document Title: Regulated PGASys Administrative Change Request Form - B


In [7]:
# ============================================================================
# CELL 7 — REF_CRO  (normalized name pool; context = description)
# ============================================================================
cro = spark.table(RAW_CRO)
CRO_NAME_COL  = "name"
CRO_DISP_COL  = "displayName"
CRO_ALIAS_COL = "alias"
CRO_DESC_COL  = "description"

def _cc(name):
    return F.col(f"`{name}`") if name in cro.columns else F.lit(None).cast("string")


desc_clean = F.when(
    F.trim(F.coalesce(_cc(CRO_DESC_COL), F.lit(""))).isin("", "1", "n/a", "N/A"),
    F.lit(None)
).otherwise(_cc(CRO_DESC_COL))

cro_enriched = (
    cro
    .withColumn("desc_clean", desc_clean)
    .filter(F.col("desc_clean").isNotNull())
    .withColumn(
        "enrichment_text",
        F.concat_ws(
            "\n",
            F.concat(F.lit("CRO (Clinical Research Organization) / System: "),
                     F.coalesce(_cc(CRO_DISP_COL), _cc(CRO_NAME_COL), F.lit(""))),
            F.concat(F.lit("Description: "), F.col("desc_clean")),
        ),
    )
    .withColumn("canonical_id", F.coalesce(_cc(CRO_DISP_COL), _cc(CRO_NAME_COL)))
)

ref_cro = (
    cro_enriched
    .withColumn("key_norm", F.explode(
        cro_variants_udf(_cc(CRO_NAME_COL), _cc(CRO_DISP_COL), _cc(CRO_ALIAS_COL))))
    .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
    .withColumn("entity_type", F.lit("CRO"))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])
)


(ref_cro.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_CRO))
spark.sql(f"COMMENT ON TABLE {REF_CRO} IS "
          "'CRO/system lookup: normalized name/display/alias variants; context = description (\\'1\\'/n/a flagged as missing).'")

# now safe to read back
cro_key_set = set(r["key_norm"] for r in spark.table(REF_CRO).select("key_norm").collect())
extract_cro_exact_udf = make_extract_cro_exact_udf(cro_key_set)
print(f"CRO known-set size: {len(cro_key_set):,}")

display(spark.table(REF_CRO).limit(20))

CRO known-set size: 6,268


,key_norm,entity_type,canonical_id,enrichment_text
0,0,CRO,GEARS,CRO (Clinical Research Organization) / System: GEARS\nDescription: FullSupport
1,0 000122844,CRO,ConfigurablePackage,CRO (Clinical Research Organization) / System: ConfigurablePackage\nDescription: Yes
2,0 00034992,CRO,ConfigurablePackage,CRO (Clinical Research Organization) / System: ConfigurablePackage\nDescription: Yes
3,0 00409962,CRO,businessCritical,CRO (Clinical Research Organization) / System: businessCritical\nDescription: G-MEDAS
4,0 042867349,CRO,COTSCommercialOffTheShelfProduct,CRO (Clinical Research Organization) / System: COTSCommercialOffTheShelfProduct\nDescription: Yes
5,0 07,CRO,Trackwise for QA + [TPUSA],CRO (Clinical Research Organization) / System: Trackwise for QA + [TPUSA]\nDescription: Yes
6,0 1,CRO,COTSCommercialOffTheShelfProduct,CRO (Clinical Research Organization) / System: COTSCommercialOffTheShelfProduct\nDescription: Yes
7,0 12,CRO,Quality Assurance Audit Database - QAAD TPUSA - v7.0.46,CRO (Clinical Research Organization) / System: Quality Assurance Audit Database - QAAD TPUSA - v7.0.46\nDescription: Yes
8,0 128041689,CRO,COTSCommercialOffTheShelfProduct,CRO (Clinical Research Organization) / System: COTSCommercialOffTheShelfProduct\nDescription: Yes
9,0 13395,CRO,ConfigurablePackage,CRO (Clinical Research Organization) / System: ConfigurablePackage\nDescription: Yes


In [8]:
# ============================================================================
# CELL 7b — REF_DEVICE  (substring name pool; context = Device_Type + business_owner)
# ============================================================================
dev = spark.table(RAW_DEVICE)
print("RAW_DEVICE columns:", dev.columns)   # <- verify names below

DEV_NAME_COL   = "Name"
DEV_ALIAS_COL  = "Alias"
DEV_LONG_COL   = "Device_Long_name"
DEV_TYPE_COL   = "Device_Type"
DEV_OWNER_COL  = "business_owner"

def _dvc(name):
    return F.col(f"`{name}`") if name in dev.columns else F.lit(None).cast("string")

dev_enriched = (
    dev
    .withColumn(
        "enrichment_text",
        F.concat_ws(
            "\n",
            F.concat(F.lit("Device: "),
                     F.coalesce(_dvc(DEV_LONG_COL), _dvc(DEV_NAME_COL), F.lit(""))),
            F.concat(F.lit("Device Type: "),
                     F.coalesce(_dvc(DEV_TYPE_COL), F.lit(""))),
            F.concat(F.lit("Business Owner: "),
                     expand_business_owner_udf(_dvc(DEV_OWNER_COL))),
        ),
    )
    .withColumn("canonical_id",
                F.coalesce(_dvc(DEV_LONG_COL), _dvc(DEV_NAME_COL)))
)

ref_device = (
    dev_enriched
    .withColumn("key_norm", F.explode(
        device_variants_udf(_dvc(DEV_NAME_COL), _dvc(DEV_ALIAS_COL), _dvc(DEV_LONG_COL))))
    .filter(F.col("key_norm").isNotNull() & (F.length("key_norm") >= 3))
    .withColumn("entity_type", F.lit("DEVICE"))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])
)

(ref_device.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_DEVICE))
spark.sql(f"COMMENT ON TABLE {REF_DEVICE} IS "
          "'Device lookup: normalized name/alias/long_name variants matched as substrings; "
          "context = Device_Type + business_owner (abbreviations expanded).'")

# read back for the substring extractor
device_key_set = set(r["key_norm"] for r in spark.table(REF_DEVICE).select("key_norm").collect())
extract_device_exact_udf = make_extract_device_exact_udf(device_key_set)
print(f"Device known-set size: {len(device_key_set):,}")

display(spark.table(REF_DEVICE).limit(20))

RAW_DEVICE columns: ['Name', 'Alias', 'Business_Owner', 'Device_Category', 'Device_Exclusivity', 'Device_Purpose', 'Device_IPOwner', 'Device_Long_Name', 'Device_Type', 'Portfolio Status', 'Partner_Alias_Link', 'Partner_Name_Link']
Device known-set size: 41


,key_norm,entity_type,canonical_id,enrichment_text
0,butterfly infusion set,DEVICE,PFD-00121,Device: PFD-00121\nDevice Type: Other Delivery System\nBusiness Owner:
1,calcheck,DEVICE,PFD-00117,Device: PFD-00117\nDevice Type: In Vitro Diagnostic (IVD)\nBusiness Owner:
2,doseguard,DEVICE,PFD-00115,Device: PFD-00115\nDevice Type: Inhaler\nBusiness Owner:
3,dual path platform (dpp),DEVICE,PFD-00116,Device: PFD-00116\nDevice Type: In Vitro Diagnostic (IVD)\nBusiness Owner:
4,hyhub ava,DEVICE,PFD-00119,Device: PFD-00119\nDevice Type: Administration Kit\nBusiness Owner:
5,hyhub duo (project pauli-us & eu),DEVICE,PFD-00120,Device: PFD-00120\nDevice Type: Administration Kit\nBusiness Owner:
6,intrathecal admin kit,DEVICE,PFD-00102,Device: PFD-00102\nDevice Type: Administration Kit\nBusiness Owner:
7,medimop,DEVICE,PFD-00107,Device: PFD-00107\nDevice Type: Vial Adapter\nBusiness Owner:
8,molly,DEVICE,PFD-00112,Device: PFD-00112\nDevice Type: Autoinjector\nBusiness Owner:
9,mypkfit hcp app,DEVICE,PFD-00110,Device: PFD-00110\nDevice Type: Software Device\nBusiness Owner:


In [9]:
# ============================================================================
# CELL 8 — REF_UNIFIED  (union of all lookups; single table to join against)
# ============================================================================
ref_unified = (
    spark.table(REF_CLINICAL).select(*LOOKUP_SCHEMA)
    .unionByName(spark.table(REF_DOCS).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_ACRONYM).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_CRO).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_DEVICE).select(*LOOKUP_SCHEMA))
    .dropDuplicates(["key_norm", "entity_type"])
)


(ref_unified.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_UNIFIED))
spark.sql(f"COMMENT ON TABLE {REF_UNIFIED} IS "
          "'Unified glossary: clinical + document + acronym + CRO + device lookups, one row per (key_norm, entity_type).'")

print("Row counts per entity_type:")
display(spark.table(REF_UNIFIED).groupBy("entity_type").count())
display(spark.table(REF_UNIFIED).limit(20))

Row counts per entity_type:


,entity_type,count
0,DOCUMENT,235073
1,DEVICE,41
2,CLINICAL_ID,18666
3,ACRONYM,1667
4,CRO,6268


,key_norm,entity_type,canonical_id,enrichment_text
0,(EU) CTR,ACRONYM,(EU) CTR,Acronym expansions: (European Union) Clinical Trials Regulation
1,(RE)-ACC1-INHIBITOR-(3238-100),CLINICAL_ID,PFP-B0123300,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism:
2,(RE)ACC2-INHIBITOR-(3448-100),CLINICAL_ID,PFP-B0125900,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism:
3,(RE)GOAT-INHIBITOR-(3714-100),CLINICAL_ID,PFP-B0191600,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Neuroscience\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: Inhibitor
4,(RE)GPR40-AGONIST-(3751-100),CLINICAL_ID,PFP-B1025800,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Others\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: Agonist
5,(RE)MEK-INHIBITOR-(3738-100),CLINICAL_ID,PFP-B1023700,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Gastroenterology\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: Inhibitor
6,(RE)MGLUR5-NAM-(3636-100),CLINICAL_ID,PFP-B0143800,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Neuroscience\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: NAM
7,(RE)OREXIN2-RECEPTOR-ANTAGONIST-(3608-100),CLINICAL_ID,PFP-B0142600,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Neuroscience\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism: Antagonist
8,(RE)SELECTIVE-ANDROGEN-RECEPTOR-MODULATOR(XVGEN)-(3350-110),CLINICAL_ID,PFP-B0106900,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Others\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism:
9,(RE)SPT-INHIBITOR-(3483-100),CLINICAL_ID,PFP-B0126400,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: \nTarget: \nMechanism:


In [10]:
# ============================================================================
# CELL 9 — SOURCE RESOLUTION  (COMPRESS TO EVENT GRAIN FIRST)
# Builds deviation_embed_input from EXACT reference matches. Cell 10 adds fuzzy CRO.
# ============================================================================
src = spark.table(SOURCE_TABLE)
print("SOURCE columns:", src.columns)

SRC_ID_COL       = "Event_Number"
SRC_PROTOCOL_COL = "Study_Protocol"
SRC_PROGRAM_COL  = "Program_Number"
SRC_DOC_COL      = "Document_or_Process"

SRC_CORE_COLS   = ["Event_Title", "Event_Description"]
SRC_EVENT_COLS  = ["Impact_Assessment", "Quality_Final_Assessment",
                   "Root_Cause_Category", "Root_Cause_SubCategory"]
SRC_ROWGRAIN_COLS = ["Action_Text"]
SRC_MID_COLS = SRC_CORE_COLS + SRC_EVENT_COLS   # mid tier = core + event (no action)


def _sc(name):
    return F.col(f"`{name}`") if name in src.columns else F.lit(None).cast("string")

# ---- 1. COMPRESS to one row per Event_Number ----
agg_exprs = []
# event-stable columns -> first non-null
for c in SRC_CORE_COLS + SRC_EVENT_COLS + [SRC_PROTOCOL_COL, SRC_PROGRAM_COL, SRC_DOC_COL]:
    if c in src.columns:
        agg_exprs.append(F.first(_sc(c), ignorenulls=True).alias(c))
# row-grain columns -> distinct NON-NULL values joined into one string
for c in SRC_ROWGRAIN_COLS:
    if c in src.columns:
        # collect_list already drops nulls; array_join avoids empty-string artifacts
        agg_exprs.append(
            F.array_join(F.array_distinct(F.collect_list(_sc(c))), "\n").alias(c)
        )

src_event = (
    src.groupBy(F.col(f"`{SRC_ID_COL}`").alias("pr_id"))
       .agg(*agg_exprs)
)
print(f"Compressed source: {src.count():,} rows -> {src_event.count():,} events")

# ---- 2. Clean + English-filter the event-grain columns ----
def _ec(name):
    return F.col(f"`{name}`") if name in src_event.columns else F.lit("").cast("string")

def _clean_english_col(name):
    return keep_if_english_udf(clean_freetext_udf(_ec(name)))

FULL_TEXT_COLS = SRC_CORE_COLS + SRC_EVENT_COLS + SRC_ROWGRAIN_COLS  # combined_text scope

src_base = (
    src_event.select(
        "pr_id",
        _ec(SRC_PROTOCOL_COL).alias("study_protocol"),
        _ec(SRC_PROGRAM_COL).alias("program_number"),
        _ec(SRC_DOC_COL).alias("document_or_process"),
        *[_clean_english_col(c).alias(f"__en_{c}") for c in FULL_TEXT_COLS],
    )
    .withColumn(
        "combined_text",   # full context: core + event + action_text
        F.concat_ws("\n",
            *[F.concat(F.lit(f"{c}: "), F.coalesce(F.col(f"__en_{c}"), F.lit("")))
              for c in FULL_TEXT_COLS]),
    )
    .withColumn(
        "core_text",       # title + description only
        F.concat_ws("\n",
            *[F.concat(F.lit(f"{c}: "), F.coalesce(F.col(f"__en_{c}"), F.lit("")))
              for c in SRC_CORE_COLS]),
    )
    .withColumn(
        "mid_text",        # core + impact + quality + root-cause (no action)
        F.concat_ws("\n",
            *[F.concat(F.lit(f"{c}: "), F.coalesce(F.col(f"__en_{c}"), F.lit("")))
              for c in SRC_MID_COLS]),
    )
    .select("pr_id", "combined_text", "mid_text", "core_text",
            "study_protocol", "program_number", "document_or_process")
    .filter(F.length(F.trim(F.col("combined_text"))) > 0)
)

# --- Extract all entity keys (free text + structured columns) ---
src_keys = (
    src_base
    .withColumn("clinical_keys",   extract_clinical_keys_udf(F.col("combined_text")))
    .withColumn("protocol_keys",   protocol_keys_udf(F.col("study_protocol")))
    .withColumn("program_keys",    program_keys_udf(F.col("program_number")))
    .withColumn("doc_keys",        extract_doc_keys_udf(F.col("combined_text")))          # free text
    .withColumn("doc_struct_keys", extract_doc_keys_udf(F.col("document_or_process")))    # structured
    .withColumn("acr_keys",        extract_acronym_keys_udf(F.col("combined_text")))
    .withColumn("cro_keys",        extract_cro_exact_udf(F.col("combined_text")))
    .withColumn("device_keys",     extract_device_exact_udf(F.col("combined_text")))
)

def _explode_keys(df, arr_col, etype):
    return (df.select("pr_id", F.explode(F.col(arr_col)).alias("key_norm"))
              .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
              .withColumn("entity_type", F.lit(etype)))

exploded = (
    _explode_keys(src_keys, "clinical_keys",   "CLINICAL_ID")
    .unionByName(_explode_keys(src_keys, "protocol_keys",   "CLINICAL_ID"))
    .unionByName(_explode_keys(src_keys, "program_keys",    "CLINICAL_ID"))
    .unionByName(_explode_keys(src_keys, "doc_keys",        "DOCUMENT"))    # free text
    .unionByName(_explode_keys(src_keys, "doc_struct_keys", "DOCUMENT"))    # structured
    .unionByName(_explode_keys(src_keys, "acr_keys",        "ACRONYM"))
    .unionByName(_explode_keys(src_keys, "cro_keys",        "CRO"))
    .unionByName(_explode_keys(src_keys, "device_keys",     "DEVICE"))
    .dropDuplicates(["pr_id", "key_norm", "entity_type"])
)

ref = spark.table(REF_UNIFIED).select(
    "key_norm", "entity_type",
    F.col("canonical_id").alias("ref_canonical_id"),
    F.col("enrichment_text").alias("ref_enrichment"),
)

resolved = exploded.join(ref, on=["key_norm", "entity_type"], how="left")

enrichment_per_pr = (
    resolved.filter(F.col("ref_enrichment").isNotNull())
    .groupBy("pr_id")
    .agg(F.concat_ws("\n---\n", F.collect_set("ref_enrichment")).alias("injected_context"))
)

def _with_ctx(text_col):
    return F.when(
        F.length(F.trim("injected_context")) > 0,
        F.concat_ws("\n\n", F.col(text_col),
            F.concat(F.lit("=== RESOLVED REFERENCES ===\n"), F.col("injected_context"))),
    ).otherwise(F.col(text_col))

embed_ready = (
    src_base.join(enrichment_per_pr, on="pr_id", how="left")
    .withColumn("injected_context", F.coalesce(F.col("injected_context"), F.lit("")))
    .withColumn("embedding_text",      _with_ctx("combined_text"))
    .withColumn("core_embedding_text", _with_ctx("core_text"))
    .withColumn("mid_embedding_text",  _with_ctx("mid_text"))
    .select("pr_id", "combined_text", "core_text", "mid_text",
            "injected_context", "embedding_text", "core_embedding_text", "mid_embedding_text")
)

# ---- Write via staging (safe on reruns where embed_input already exists) ----
EMBED_INPUT = f"{CATALOG}.{ALYT}.deviation_embed_input"
STAGING     = f"{CATALOG}.{ALYT}.deviation_embed_input_staging"

(embed_ready.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STAGING))
(spark.table(STAGING).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(EMBED_INPUT))
spark.sql(f"DROP TABLE IF EXISTS {STAGING}")

print("Coverage — deviations with >=1 resolved reference:")
display(embed_ready.select(
    F.count("*").alias("total"),
    F.sum(F.when(F.length("injected_context") > 0, 1).otherwise(0)).alias("with_context"),
))
display(spark.table(EMBED_INPUT).limit(10))

SOURCE columns: ['Event_Number', 'Source_System', 'Internal_or_External', 'Event_Status', 'Event_Rating', 'Event_Type', 'Event_Owner', 'QA_Contact', 'Discovery_Date', 'Event_Description', 'Event_Title', 'Initial_Event_Rating', 'Impact_Assessment', 'Document_or_Process', 'Dev_Event_Scope_Other', 'Event_Closed_Date', 'Owning_Department', 'Therapeutic_Area', 'Program_Name', 'Program_Number', 'Study_Protocol', 'Study_Phase', 'Data_Load_Date', 'CAPA_Parent_ID', 'Quality_Approver_CAPA_Final', 'CAPA_Parent_Opened_Date', 'QA_Approval_for_Cancelation_On', 'Action_Number', 'State_CAPA_Final', 'Action_Text', 'Action_Owner', 'Action_Due_Date', 'Action_Completed_Date', 'Action_Status', 'EC_Number', 'EC_Due_Date', 'EC_Completed_Date', 'EC_Text', 'EC_Owner', 'EC_Status', 'EC_Result', 'Regulatory_Focus', 'Local_List_B', 'Recurring', 'Reported_by', 'Date_Severity_Approved', 'Date_Reported_to_QAC', 'CAPA_Plan_Approval_Date', 'CRO', 'Date_of_Occurrence', 'PV_System_or_PSMF', 'CRO_Deviation', 'Root_Cause_

,total,with_context
0,3778,1876


,pr_id,combined_text,core_text,mid_text,injected_context,embedding_text,core_embedding_text,mid_embedding_text
0,1901905,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory: \nAction_Text:,Event_Title: \nEvent_Description:,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory:,Generic: \nModality: Biologics\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: Support - Oncology\nTarget: \nMechanism:,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory: \nAction_Text: \n\n=== RESOLVED REFERENCES ===\nGeneric: \nModality: Biologics\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: Support - Oncology\nTarget: \nMechanism:,Event_Title: \nEvent_Description: \n\n=== RESOLVED REFERENCES ===\nGeneric: \nModality: Biologics\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: Support - Oncology\nTarget: \nMechanism:,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory: \n\n=== RESOLVED REFERENCES ===\nGeneric: \nModality: Biologics\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: Support - Oncology\nTarget: \nMechanism:
1,2013213,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory: \nAction_Text:,Event_Title: \nEvent_Description:,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory:,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: RGH\nFinance TA Grouping: \nIndication: Post-Transplant CMV\nTarget: \nMechanism: Inhibitor,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory: \nAction_Text: \n\n=== RESOLVED REFERENCES ===\nGeneric: \nModality: Synthetic Molecules\nTherapeutic Area: RGH\nFinance TA Grouping: \nIndication: Post-Transplant CMV\nTarget: \nMechanism: Inhibitor,Event_Title: \nEvent_Description: \n\n=== RESOLVED REFERENCES ===\nGeneric: \nModality: Synthetic Molecules\nTherapeutic Area: RGH\nFinance TA Grouping: \nIndication: Post-Transplant CMV\nTarget: \nMechanism: Inhibitor,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory: \n\n=== RESOLVED REFERENCES ===\nGeneric: \nModality: Synthetic Molecules\nTherapeutic Area: RGH\nFinance TA Grouping: \nIndication: Post-Transplant CMV\nTarget: \nMechanism: Inhibitor
2,2066324,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory: \nAction_Text:,Event_Title: \nEvent_Description:,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory:,Generic: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: (CML) Chronic myeloid leukemia\nTarget: \nMechanism: Inhibitor\n---\nGeneric: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: (Ph+ ALL) Philadelphia chromosome-positive acute lymphoblastic leukemia\nTarget: \nMechanism: Inhibitor,Event_Title: \nEvent_Description: \nImpact_Assessment: \nQuality_Final_Assessment: \nRoot_Cause_Category: \nRoot_Cause_SubCategory: \nAction_Text: \n\n=== RESOLVED REFERENCES ===\nGeneric: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: (CML) Chronic myeloid leukemia\nTarget: \nMechanism: Inhibitor\n---\nGeneric: \nModality: Synthetic Molecules\nTherapeutic Area: Oncology\nFinance TA Grouping: \nIndication: (Ph+ ALL) Philadelphia chromosome-positive acute lymphoblastic leukemia\nTarget: \nMechanism: Inhibito

In [17]:
# DIAGNOSTIC — how many source rows extract each entity type, and how many join
print("Source rows with ≥1 extracted key (pre-join):")
display(src_keys.select(
    F.sum(F.when(F.size("clinical_keys") > 0, 1).otherwise(0)).alias("has_clinical"),
    F.sum(F.when((F.size("doc_keys") > 0) | (F.size("doc_struct_keys") > 0), 1).otherwise(0)).alias("has_doc"),
    F.sum(F.when(F.size("acr_keys") > 0, 1).otherwise(0)).alias("has_acr"),
    F.sum(F.when(F.size("cro_keys") > 0, 1).otherwise(0)).alias("has_cro_exact"),   # <-- added
))

print("Exploded keys that actually matched REF_UNIFIED:")
display(
    exploded.join(ref, on=["key_norm", "entity_type"], how="left")
    .groupBy("entity_type")
    .agg(
        F.count("*").alias("extracted"),
        F.sum(F.when(F.col("ref_enrichment").isNotNull(), 1).otherwise(0)).alias("matched"),
    )
)

print("Top extracted keys that FAILED to match (fix these first):")
display(
    exploded.join(ref, on=["key_norm", "entity_type"], how="left")
    .filter(F.col("ref_enrichment").isNull())
    .groupBy("entity_type", "key_norm").count()
    .orderBy(F.col("count").desc()).limit(30)
)

Source rows with ≥1 extracted key (pre-join):


,has_clinical,has_doc,has_acr,has_cro_exact
0,0,999,64,0


Exploded keys that actually matched REF_UNIFIED:


,entity_type,extracted,matched
0,DOCUMENT,1418,1172
1,CLINICAL_ID,2658,2348
2,ACRONYM,64,64


Top extracted keys that FAILED to match (fix these first):


,entity_type,key_norm,count
0,DOCUMENT,SOP-217615,22
1,DOCUMENT,PROC-0003653,18
2,DOCUMENT,TOOL-0002107,17
3,CLINICAL_ID,N-A,14
4,DOCUMENT,PROC-0004197,14
5,DOCUMENT,TOOL-0001858,13
6,DOCUMENT,SOP-218635,8
7,CLINICAL_ID,FRUQ-NPP,8
8,DOCUMENT,TOOL-224362,7
9,DOCUMENT,SOP-252083,6
